In [1]:
using NumericalEarth
using Oceananigans
using Oceananigans.Units
using Oceananigans.Grids: node
using Oceananigans.TurbulenceClosures:
    IsopycnalSkewSymmetricDiffusivity,
    AdvectiveFormulation
using Dates
using Printf
using CUDA

In [2]:
# determining run version/params - :validation, :spinup, or :production
run_mode = :validation

@assert run_mode in (:validation, :spinup, :production)

validation_days = 30
spinup_days = 365
production_days = 180

use_rivers = true

# start at 90 seconds; TimeStepWizard may decrease or increase this
initial_time_step = 90seconds

# maximum timestep allowed by TimeStepWizard
maximum_time_step = 180seconds

resolution_tag = "0p05deg_nz20_gm_on_redi_off_trial4_10_salinity"

if use_rivers
    run_name = "rivers_on"
else
    run_name = "rivers_off"
end

if run_mode == :validation
    output_tag = "$(run_name)_validation_$(resolution_tag)"
elseif run_mode == :spinup
    output_tag = "$(run_name)_spinup_$(resolution_tag)"
elseif run_mode == :production
    output_tag = "$(run_name)_production_$(resolution_tag)"
end

surface_filename = "amazon_$(output_tag)_surface_fields"
free_surface_filename = "amazon_$(output_tag)_free_surface"
dye_3d_filename = "amazon_$(output_tag)_dye_3d"
salinity_3d_filename = "amazon_$(output_tag)_salinity_3d"

checkpoint_prefix = "amazon_$(run_name)_$(resolution_tag)_$(run_mode)_checkpoint"

production_initial_checkpoint = "amazon_$(run_name)_$(resolution_tag)_day365_with_dye.jld2"

"amazon_rivers_on_0p05deg_nz20_gm_on_redi_off_trial4_10_salinity_day365_with_dye.jld2"

In [3]:
arch = GPU()

# grid definitions -- amazon river mouth / plume region
long_west = -58.5
long_east = -41.5
lat_south = -7.8
lat_north = 9.2

Nx = 340
Ny = 340
Nz = 20

long_river = -49.5
lat_river = 0.16

depth = 4000meters

longitude_resolution = (long_east - long_west) / Nx
latitude_resolution = (lat_north - lat_south) / Ny

@assert longitude_resolution ≈ 0.05
@assert latitude_resolution ≈ 0.05

z = ExponentialDiscretization(
    Nz,
    -depth,
    0;
    scale = depth / 4,
    mutable = false
)

underlying_grid = LatitudeLongitudeGrid(
    arch;
    size = (Nx, Ny, Nz),
    halo = (5, 5, 4),
    longitude = (long_west, long_east),
    latitude = (lat_south, lat_north),
    z,
    topology = (Bounded, Bounded, Bounded)
)

# use at least 30 m so river-mouth columns can mix below the surface cell
bottom_height = regrid_bathymetry(
    underlying_grid;
    minimum_depth = 30,
    interpolation_passes = 10,
    major_basins = 1
)

grid = ImmersedBoundaryGrid(
    underlying_grid,
    GridFittedBottom(bottom_height);
    active_cells_map = true
)

[ Info: Loading cached bathymetry from C:\Users\meghn\.julia\scratchspaces\904d977b-046a-4731-8b86-9235c0d1ef02\bathymetry_cache\bathymetry_340x340_-58.5_-41.5_-7.799999999999999_9.2_51edf445.jld2


340×340×20 ImmersedBoundaryGrid{Float64, Bounded, Bounded, Bounded} on CUDAGPU with 5×5×4 halo:
├── immersed_boundary: GridFittedBottom(mean(z)=-1077.73, min(z)=-4000.0, max(z)=0.0)
├── underlying_grid: 340×340×20 LatitudeLongitudeGrid{Float64, Bounded, Bounded, Bounded} on CUDAGPU with 5×5×4 halo
├── longitude: Bounded  λ ∈ [-58.5, -41.5] regularly spaced with Δλ=0.05
├── latitude:  Bounded  φ ∈ [-7.8, 9.2]    regularly spaced with Δφ=0.05
└── z:         Bounded  z ∈ [-4000.0, 0.0] variably spaced with min(Δz)=16.5232, max(Δz)=738.605

In [4]:
# Gent–McWilliams remains on.
# Redi symmetric diffusion is zero for every tracer.
const gm_skew_diffusivity = 1e3
const redi_diffusivity = 0.0

eddy_closure = IsopycnalSkewSymmetricDiffusivity(
    κ_skew = gm_skew_diffusivity,
    κ_symmetric = (
        T = redi_diffusivity,
        S = redi_diffusivity,
        dye = redi_diffusivity,
        e = redi_diffusivity
    ),
    skew_flux_formulation = AdvectiveFormulation()
)

# default NumericalEarth vertical mixing
vertical_mixing = NumericalEarth.Oceans.default_ocean_closure()

# Trial 4.10 additional vertical tracer mixing near the river mouth
const river_mixing_longitude = -49.35
const river_mixing_latitude = -0.15
const river_mixing_kz = 1e-3
const river_mixing_depth = 30meters
const river_mixing_half_width = 0.75

@inline function river_mouth_kz(longitude, latitude, z, time)
    inside_longitude = abs(longitude - river_mixing_longitude) <= river_mixing_half_width
    inside_latitude = abs(latitude - river_mixing_latitude) <= river_mixing_half_width
    inside_depth = z >= -river_mixing_depth

    if inside_longitude && inside_latitude && inside_depth
        return river_mixing_kz
    else
        return 0.0
    end
end

river_mixing = VerticalScalarDiffusivity(
    VerticallyImplicitTimeDiscretization();
    κ = (
        T = river_mouth_kz,
        S = river_mouth_kz,
        dye = river_mouth_kz,
        e = 0.0
    )
)

closure = (
    vertical_mixing,
    river_mixing
)

@assert gm_skew_diffusivity == 1e3
@assert redi_diffusivity == 0.0

@assert river_mouth_kz(
    river_mixing_longitude,
    river_mixing_latitude,
    -10,
    0
) == river_mixing_kz

@assert river_mouth_kz(
    river_mixing_longitude,
    river_mixing_latitude,
    -40,
    0
) == 0.0

In [ ]:
free_surface = SplitExplicitFreeSurface(
    grid;
    substeps = 140
)

momentum_advection = WENOVectorInvariant(order = 5)

# preserve Trial 4.10 tracer advection
tracer_advection = WENO(order = 5)

# create zero-gradient boundary conditions for dye
dye_bcs = FieldBoundaryConditions(
    west = GradientBoundaryCondition(0),
    east = GradientBoundaryCondition(0),
    south = GradientBoundaryCondition(0),
    north = GradientBoundaryCondition(0),
    top = FluxBoundaryCondition(0),
    bottom = FluxBoundaryCondition(0)
)

model_bcs = (
    dye = dye_bcs,
)

# these values must be constant because the sponge runs on the GPU
const north_sponge_mask = GaussianMask{:y}(
    center = 9.2,
    width = 0.25
)

const north_sponge_rate = 1 / 5days

@inline function north_dye_sponge(i, j, k, grid, clock, model_fields)
    x, y, z = node(
        i,
        j,
        k,
        grid,
        Center(),
        Center(),
        Center()
    )

    mask = north_sponge_mask(x, y, z)

    dye = @inbounds model_fields.dye[i, j, k]

    return -north_sponge_rate * mask * dye
end

dye_sponge = Forcing(
    north_dye_sponge;
    discrete_form = true
)

model_forcing = (
    dye = dye_sponge,
)

ocean = ocean_simulation(
    grid;
    momentum_advection,
    tracer_advection,
    free_surface,
    closure = closure,
    tracers = (:T, :S, :dye),
    boundary_conditions = model_bcs,
    forcing = model_forcing
)

@show ocean.model.closure

LoadError: AssertionError: length(ocean.model.closure) == 3

In [ ]:
@inline function dye_initial_condition(x, y, z)
    horizontal_width = 0.5

    horizontal_blob = exp(
        -(
            (x - long_river)^2 +
            (y - lat_river)^2
        ) / horizontal_width^2
    )

    if z > -10
        return horizontal_blob
    else
        return 0.0
    end
end

# validation and spinup are fresh runs initialized from ECCO
if run_mode == :validation || run_mode == :spinup
    ENV["ECCO_USERNAME"] = "meggoeggo"
    ENV["ECCO_WEBDAV_PASSWORD"] = "3a0bqyAsTojLw4ZtpKaC"


    date = DateTime(1993, 1, 1)

    ecco_variables = (
        :temperature,
        :salinity
    )

    ecco_set = MetadataSet(
        ecco_variables;
        dataset = ECCO4Monthly(),
        date
    )

    set!(ocean.model, ecco_set)
end

land = JRA55PrescribedLand(arch)

atmosphere = JRA55PrescribedAtmosphere(arch)

ocean_surface = SurfaceRadiationProperties(
    albedo = LatitudeDependentAlbedo()
)

radiation = JRA55PrescribedRadiation(
    arch;
    ocean_surface
)

@assert use_rivers "The final model requires the original JRA55 river forcing"

coupled_model = EarthSystemModel(
    ;
    ocean,
    atmosphere,
    land,
    radiation
)

if run_mode == :validation
    stop_day = validation_days
elseif run_mode == :spinup
    stop_day = spinup_days
elseif run_mode == :production
    # production begins at day 365 and runs for another 180 days
    stop_day = spinup_days + production_days
end

simulation = Simulation(
    coupled_model;
    Δt = initial_time_step,
    stop_time = stop_day * days
)

# TimeStepWizard is attached to the coupled EarthSystemModel.
# Calculate its advective timescale from the ocean component.
function ocean_advection_timescale(earth_system)
    return Oceananigans.Advection.cell_advection_timescale(
        earth_system.ocean.model
    )
end

time_step_wizard = TimeStepWizard(
    cfl = 0.5,
    max_change = 1.05,
    min_change = 0.2,
    max_Δt = maximum_time_step, 
    min_Δt = 1.0,
    cell_advection_timescale = ocean_advection_timescale
)

simulation.callbacks[:time_step_wizard] = Callback(
    time_step_wizard,
    IterationInterval(5)
)

┌ Warning: 1440×720×1×365 PrescribedLand tracks time as Float32 but the EarthSystemModel clock uses Float64; coercing the component clock to keep components synchronized.
└ @ NumericalEarth.EarthSystemModels C:\Users\meghn\.julia\packages\NumericalEarth\eKhWQ\src\EarthSystemModels\components.jl:136


Callback of TimeStepWizard(cfl=0.5, max_Δt=180.0, min_Δt=1.0) on IterationInterval(5)

In [ ]:
wall_time = Ref(time_ns())

function progress(sim)
    ocean = sim.model.ocean

    u, v, w = ocean.model.velocities

    T = ocean.model.tracers.T
    S = ocean.model.tracers.S
    dye = ocean.model.tracers.dye
    e = ocean.model.tracers.e

    Tmin, Tmax = minimum(T), maximum(T)
    Smin, Smax = minimum(S), maximum(S)
    dyemin, dyemax = minimum(dye), maximum(dye)
    emin, emax = minimum(e), maximum(e)

    umax = (
        maximum(abs, u),
        maximum(abs, v),
        maximum(abs, w)
    )

    current_Δt = sim.Δt

    cfl = AdvectiveCFL(current_Δt)(
        ocean.model
    )

    step_time = 1e-9 * (
        time_ns() - wall_time[]
    )

    msg1 = @sprintf(
        "time: %s, iter: %d, Δt: %s",
        prettytime(sim),
        iteration(sim),
        prettytime(current_Δt)
    )

    msg2 = @sprintf(
        ", max|u|: (%.3e, %.3e, %.3e) m s⁻¹",
        umax[1],
        umax[2],
        umax[3]
    )

    msg3 = @sprintf(
        "\nT:    min = %.6e, max = %.6e °C",
        Tmin,
        Tmax
    )

    msg4 = @sprintf(
        "\nS:    min = %.6e, max = %.6e",
        Smin,
        Smax
    )

    msg5 = @sprintf(
        "\ndye:  min = %.6e, max = %.6e",
        dyemin,
        dyemax
    )

    msg6 = @sprintf(
        "\ne:    min = %.6e, max = %.6e m² s⁻²",
        emin,
        emax
    )

    msg7 = @sprintf(
        "\nadvective CFL = %.6e",
        cfl
    )

    msg8 = @sprintf(
        "\nwall time since last report: %s\n",
        prettytime(step_time)
    )

    @info msg1 * msg2 * msg3 * msg4 * msg5 * msg6 * msg7 * msg8

    if dyemin < 0
        warning_message = @sprintf(
            "NEGATIVE DYE DETECTED: min(dye) = %.6e",
            dyemin
        )

        @warn warning_message
    end

    if dyemax > 1
        warning_message = @sprintf(
            "DYE OVERSHOOT DETECTED: max(dye) = %.6e",
            dyemax
        )

        @warn warning_message
    end

    if Smin < 0
        warning_message = @sprintf(
            "NEGATIVE SALINITY DETECTED: min(S) = %.6e",
            Smin
        )

        @warn warning_message
    end

    if cfl > 0.8
        warning_message = @sprintf(
            "ADVECTIVE CFL ABOVE 0.8: CFL = %.6e",
            cfl
        )

        @warn warning_message
    end

    wall_time[] = time_ns()

    return nothing
end

add_callback!(
    simulation,
    progress,
    TimeInterval(1days)
)

In [ ]:
# collect surface tracers and velocities
ocean_outputs = merge(
    ocean.model.tracers,
    ocean.model.velocities
)

# grab the free-surface displacement
free_surface = ocean.model.free_surface.displacement

# save daily surface tracers and velocities
ocean.output_writers[:surface] = JLD2Writer(
    ocean.model,
    ocean_outputs;
    schedule = TimeInterval(1days),
    filename = surface_filename,
    indices = (:, :, grid.Nz),
    overwrite_existing = true
)

# save daily sea-surface height
ocean.output_writers[:free_surface] = JLD2Writer(
    ocean.model,
    (
        η = free_surface,
    );
    schedule = TimeInterval(1days),
    filename = free_surface_filename,
    overwrite_existing = true
)

if run_mode == :validation
    salinity_output_interval = 1days
elseif run_mode == :spinup
    salinity_output_interval = 30days
elseif run_mode == :production
    salinity_output_interval = 5days
end

ocean.output_writers[:salinity_3d] = JLD2Writer(
    ocean.model,
    (
        S = ocean.model.tracers.S,
    );
    schedule = TimeInterval(salinity_output_interval),
    filename = salinity_3d_filename,
    overwrite_existing = true
)

if run_mode == :production
    ocean.output_writers[:dye_3d] = JLD2Writer(
        ocean.model,
        (
            dye = ocean.model.tracers.dye,
        );
        schedule = TimeInterval(1days),
        filename = dye_3d_filename,
        overwrite_existing = true
    )
end

if run_mode == :spinup || run_mode == :production
    simulation.output_writers[:checkpointer] = Checkpointer(
        simulation.model;
        schedule = TimeInterval(30days),
        prefix = checkpoint_prefix,
        cleanup = true,
        overwrite_existing = true
    )
end

In [ ]:
@assert iteration(simulation) == 0 "Restart the kernel and rebuild the simulation before starting a run"

if run_mode == :validation
    # validation tests salinity without dye
    set!(
        ocean.model,
        dye = 0.0
    )

    @info "Starting 60-day salinity validation"

    progress(simulation)

    run!(simulation)

    validation_Smin = minimum(
        ocean.model.tracers.S
    )

    validation_Smax = maximum(
        ocean.model.tracers.S
    )

    validation_Δt = simulation.Δt

    validation_CFL = AdvectiveCFL(
        validation_Δt
    )(
        ocean.model
    )

    @info(
        "60-day validation complete",
        validation_Smin,
        validation_Smax,
        validation_CFL,
        validation_Δt
    )

    if validation_Smin < 0
        error(
            "Validation failed: negative salinity = $validation_Smin"
        )
    end

    if validation_CFL > 0.8
        error(
            "Validation failed: final advective CFL = $validation_CFL"
        )
    end

elseif run_mode == :spinup
    # spinup begins from ECCO with no dye
    set!(
        ocean.model,
        dye = 0.0
    )

    @assert !isfile(
        production_initial_checkpoint
    ) "Move or rename the existing production checkpoint before starting a new spinup"

    @info "Starting fresh 365-day spinup"

    progress(simulation)

    run!(
        simulation;
        checkpoint_at_end = true
    )

    spinup_Smin = minimum(
        ocean.model.tracers.S
    )

    spinup_Smax = maximum(
        ocean.model.tracers.S
    )

    @info(
        "365-day spinup complete",
        spinup_Smin,
        spinup_Smax,
        final_Δt = simulation.Δt
    )

    if spinup_Smin < 0
        error(
            "Spinup produced negative salinity: $spinup_Smin"
        )
    end

    # deposit dye only after spinup
    set!(
        ocean.model,
        dye = dye_initial_condition
    )

    minimum_dye = minimum(
        ocean.model.tracers.dye
    )

    maximum_dye = maximum(
        ocean.model.tracers.dye
    )

    @info(
        "Dye deposited after spinup",
        minimum_dye,
        maximum_dye
    )

    # save the spun-up state with newly deposited dye
    checkpoint(
        simulation;
        filepath = production_initial_checkpoint
    )

    @info(
        "Production checkpoint written",
        production_initial_checkpoint
    )

elseif run_mode == :production
    @assert isfile(
        production_initial_checkpoint
    ) "Run the 365-day spinup first; production checkpoint is missing"

    @info(
        "Starting 180-day dye production run",
        production_initial_checkpoint
    )

    # pickup restores the day-365 clock and dye state
    # the simulation stops at absolute model day 545
    run!(
        simulation;
        pickup = production_initial_checkpoint,
        checkpoint_at_end = true
    )
end

[ Info: Starting 60-day salinity validation
┌ Info: time: 0 seconds, iter: 0, Δt: 1.500 minutes, max|u|: (0.000e+00, 0.000e+00, 0.000e+00) m s⁻¹
│ T:    min = 2.119263e+00, max = 2.817149e+01 °C
│ S:    min = 3.102885e+01, max = 3.669934e+01
│ dye:  min = 0.000000e+00, max = 0.000000e+00
│ e:    min = 0.000000e+00, max = 0.000000e+00 m² s⁻²
│ advective CFL = 0.000000e+00
└ wall time since last report: 33.245 seconds
[ Info: Initializing simulation...
┌ Info: time: 0 seconds, iter: 0, Δt: 1.575 minutes, max|u|: (0.000e+00, 0.000e+00, 0.000e+00) m s⁻¹
│ T:    min = 2.119263e+00, max = 2.817149e+01 °C
│ S:    min = 3.102885e+01, max = 3.669934e+01
│ dye:  min = 0.000000e+00, max = 0.000000e+00
│ e:    min = 0.000000e+00, max = 0.000000e+00 m² s⁻²
│ advective CFL = 0.000000e+00
└ wall time since last report: 11.255 seconds
[ Info:     ... simulation initialization complete (2.342 seconds)
[ Info: Executing initial time step...
[ Info:     ... initial time step complete (2.876 minutes).
┌ I